
## Домашнее задание 4. Трансформеры. 

In [81]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertModel, BertTokenizer

## Подготовка данных

In [82]:
from datasets import load_dataset


dataset = load_dataset("IlyaGusev/gazeta", split="train[:10%]")
val_dataset = load_dataset("IlyaGusev/gazeta", split="validation[:5%]")

In [83]:
model_name = "ai-forever/ruBert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)


def preprocess(examples, use_padding=True):
    model_inputs = tokenizer(
        examples["text"],
        max_length=224,
        padding="max_length" if use_padding else False,
        truncation=True,
    )

    labels = tokenizer(
        examples["summary"],
        max_length=80,
        padding="max_length" if use_padding else False,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [84]:
tokenized_dataset = dataset.map(preprocess, batched=False)
tokenized_dataset.set_format("torch")

tokenized_val_dataset = val_dataset.map(preprocess, batched=False)
tokenized_val_dataset.set_format("torch")

Map:   0%|          | 0/6096 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

In [85]:
from torch.utils.data import DataLoader
import os


def collate_batch(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
    }


num_workers = 0 if os.name == "nt" else 2
loader_kwargs = {
    "num_workers": num_workers,
    "pin_memory": torch.cuda.is_available(),
}
if loader_kwargs["num_workers"] > 0:
    loader_kwargs["persistent_workers"] = True

train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_batch,
    **loader_kwargs,
)

eval_dataloader = DataLoader(
    tokenized_val_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
    **loader_kwargs,
)

print(f"Батчей в трейне: {len(train_dataloader)}")
print(f"Батчей в валидации: {len(eval_dataloader)}")

Батчей в трейне: 762
Батчей в валидации: 40


## Реализация Decoder-cети 

In [86]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel


class BertSummarizer(nn.Module):
    def __init__(
        self,
        bert_model_name="ai-forever/ruBert-base",
        hidden_size=768,
        num_decoder_layers=2,
        num_heads=8,
        dropout=0.1,
        pad_token_id=0,
    ):
        super().__init__()

        self.bert = BertModel.from_pretrained(bert_model_name)
        self.hidden_size = hidden_size
        self.pad_token_id = pad_token_id
        self.embedding = self.bert.embeddings.word_embeddings
        self.pos_embedding = nn.Embedding(512, hidden_size)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            batch_first=False,
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer, num_layers=num_decoder_layers
        )

        self.fc_out = nn.Linear(hidden_size, self.bert.config.vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if "bert" in name or "fc_out" in name:
                continue
            if "pos_embedding" in name:
                nn.init.normal_(p, mean=0, std=0.02)
            elif p.dim() > 1:
                nn.init.xavier_uniform_(p)
            else:
                nn.init.zeros_(p)

    def generate_square_subsequent_mask(self, T):
        return torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

    def forward(self, input_ids, attention_mask, decoder_input_ids):
        encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        memory = encoder_outputs.last_hidden_state.transpose(0, 1)  # (S, B, D)

        B, T = decoder_input_ids.shape
        positions = torch.arange(T, device=decoder_input_ids.device).unsqueeze(0).expand(B, -1)
        embedded = self.embedding(decoder_input_ids) + self.pos_embedding(positions)
        embedded = embedded.transpose(0, 1)  # (T, B, D)

        tgt_mask = self.generate_square_subsequent_mask(T).to(input_ids.device)
        memory_key_padding_mask = attention_mask == 0
        tgt_key_padding_mask = decoder_input_ids == self.pad_token_id

        decoder_output = self.decoder(
            tgt=embedded,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )

        logits = self.fc_out(decoder_output.transpose(0, 1))
        return logits

    def generate(
        self,
        input_ids,
        attention_mask,
        tokenizer,
        max_len=80,
        strategy="greedy",
        top_k=50,
        top_p=0.9,
        temperature=1.0,
        num_beams=4,
        min_len=20,
        repetition_penalty=1.2,
    ):
        if strategy == "beam":
            return self._beam_search(
                input_ids,
                attention_mask,
                tokenizer,
                max_len,
                num_beams,
                repetition_penalty=3.0,
            )

        self.eval()
        device = input_ids.device
        dot_token_id = tokenizer.convert_tokens_to_ids(".")

        with torch.no_grad():
            encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            memory = encoder_outputs.last_hidden_state.transpose(0, 1)
            memory_key_padding_mask = attention_mask == 0

            batch_size = input_ids.size(0)
            decoder_input_ids = torch.full(
                (batch_size, 1),
                tokenizer.cls_token_id,
                dtype=torch.long,
                device=device,
            )

            for step in range(max_len):
                T = decoder_input_ids.size(1)
                positions = torch.arange(T, device=device).unsqueeze(0).expand(batch_size, -1)
                embedded = (
                    self.embedding(decoder_input_ids) + self.pos_embedding(positions)
                ).transpose(0, 1)

                tgt_mask = self.generate_square_subsequent_mask(T).to(device)

                decoder_output = self.decoder(
                    tgt=embedded,
                    memory=memory,
                    tgt_mask=tgt_mask,
                    memory_key_padding_mask=memory_key_padding_mask,
                )

                logits = self.fc_out(decoder_output[-1, :, :])
                logits = logits / max(temperature, 1e-5)

                for b in range(batch_size):
                    used_tokens = decoder_input_ids[b].unique()
                    logits[b, used_tokens] = logits[b, used_tokens] / repetition_penalty

                    if step < min_len:
                        logits[b, tokenizer.sep_token_id] = -float("inf")

                    if decoder_input_ids.size(1) >= 2 and decoder_input_ids[b, -1].item() == dot_token_id:
                        logits[b, dot_token_id] = -float("inf")

                if strategy == "greedy":
                    next_token = torch.argmax(logits, dim=-1, keepdim=True)
                elif strategy == "top_k":
                    next_token = self._top_k_sampling(logits, top_k)
                elif strategy == "top_p":
                    next_token = self._top_p_sampling(logits, top_p)
                else:
                    raise ValueError(f"Unknown strategy: {strategy}")

                decoder_input_ids = torch.cat([decoder_input_ids, next_token], dim=1)

                if (next_token == tokenizer.sep_token_id).all():
                    break

        return tokenizer.decode(
            decoder_input_ids.squeeze().tolist(),
            skip_special_tokens=True,
        )

    def _top_k_sampling(self, logits, top_k):
        top_k_logits, _ = torch.topk(logits, top_k, dim=-1)
        threshold = top_k_logits[:, -1].unsqueeze(-1)
        filtered_logits = logits.masked_fill(logits < threshold, float("-inf"))
        probs = F.softmax(filtered_logits, dim=-1)
        return torch.multinomial(probs, num_samples=1)

    def _top_p_sampling(self, logits, top_p):
        probs = F.softmax(logits, dim=-1)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = cumulative_probs - sorted_probs > top_p

        sorted_logits = logits.gather(-1, sorted_indices)
        sorted_logits[remove_mask] = float("-inf")
        filtered_logits = sorted_logits.scatter(-1, sorted_indices, sorted_logits)

        probs = F.softmax(filtered_logits, dim=-1)
        return torch.multinomial(probs, num_samples=1)

    def _beam_search(
        self,
        input_ids,
        attention_mask,
        tokenizer,
        max_len,
        num_beams,
        repetition_penalty=3.0,
    ):
        self.eval()
        device = input_ids.device

        with torch.no_grad():
            assert input_ids.size(0) == 1

            encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            memory = encoder_outputs.last_hidden_state.transpose(0, 1)
            memory = memory.expand(-1, num_beams, -1)
            memory_kpm = (attention_mask == 0).expand(num_beams, -1)

            beams = [(0.0, [tokenizer.cls_token_id])]
            completed = []

            for _ in range(max_len):
                all_candidates = []

                beam_tokens = torch.tensor([b[1] for b in beams], dtype=torch.long, device=device)
                B_cur, T = beam_tokens.shape
                positions = torch.arange(T, device=device).unsqueeze(0).expand(B_cur, -1)
                embedded = (
                    self.embedding(beam_tokens) + self.pos_embedding(positions)
                ).transpose(0, 1)

                mem = memory[:, :B_cur, :]
                mem_kpm = memory_kpm[:B_cur, :]
                tgt_mask = self.generate_square_subsequent_mask(T).to(device)

                decoder_output = self.decoder(
                    tgt=embedded,
                    memory=mem,
                    tgt_mask=tgt_mask,
                    memory_key_padding_mask=mem_kpm,
                )
                logits = self.fc_out(decoder_output[-1, :, :])

                for i, (_, tokens) in enumerate(beams):
                    for token_id in set(tokens):
                        logits[i, token_id] /= repetition_penalty

                log_probs = F.log_softmax(logits, dim=-1)

                for i, (score, tokens) in enumerate(beams):
                    top_log_probs, top_indices = log_probs[i].topk(num_beams)
                    for log_p, token_id in zip(top_log_probs.tolist(), top_indices.tolist()):
                        new_score = score + log_p
                        new_tokens = tokens + [token_id]
                        if token_id == tokenizer.sep_token_id:
                            completed.append((new_score / len(new_tokens), new_tokens))
                        else:
                            all_candidates.append((new_score, new_tokens))

                if not all_candidates:
                    break

                all_candidates.sort(key=lambda x: x[0], reverse=True)
                beams = all_candidates[:num_beams]

            for score, tokens in beams:
                completed.append((score / len(tokens), tokens))

            completed.sort(key=lambda x: x[0], reverse=True)
            best_tokens = completed[0][1]

        return tokenizer.decode(best_tokens, skip_special_tokens=True)


In [87]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertSummarizer(bert_model_name=model_name, pad_token_id=tokenizer.pad_token_id)
model = model.to(device)
print(f"Vocab size: {model.bert.config.vocab_size}")
print(f"Device: {device}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vocab size: 120138
Device: cuda


In [88]:
# Проверяем генерацию до обучения
sample = next(iter(eval_dataloader))
print("Генерация до обучения:")
print(model.generate(
    sample["input_ids"][:1].to(device),
    sample["attention_mask"][:1].to(device),
    tokenizer,
))

Генерация до обучения:



## Обучение модели

In [89]:
def train_step(model, input_ids, attention_mask, decoder_input_ids, optimizer, criterion, scaler=None, use_amp=False):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    dec_input = decoder_input_ids[:, :-1]
    targets = decoder_input_ids[:, 1:]

    with torch.amp.autocast("cuda", enabled=use_amp):
        logits = model(input_ids, attention_mask, dec_input)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
        )

    if scaler is not None and use_amp:
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    return loss.item()


def eval_step(model, input_ids, attention_mask, decoder_input_ids, criterion, use_amp=False):
    model.eval()
    with torch.no_grad():
        dec_input = decoder_input_ids[:, :-1]
        targets = decoder_input_ids[:, 1:]

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(input_ids, attention_mask, dec_input)
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
    return loss.item()


In [91]:
from torch.optim import AdamW
from tqdm import tqdm

NUM_EPOCHS = 8
FREEZE_EPOCHS = 3

for param in model.bert.parameters():
    param.requires_grad = False

optimizer = AdamW([
    {"params": model.pos_embedding.parameters(), "lr": 5e-4},
    {"params": model.decoder.parameters(), "lr": 5e-4},
], weight_decay=0.01)

criterion = nn.CrossEntropyLoss(
    ignore_index=tokenizer.pad_token_id,
    label_smoothing=0.1,
)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

train_losses = []
val_losses = []
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    if epoch == FREEZE_EPOCHS:
        for param in model.bert.embeddings.parameters():
            param.requires_grad = True
        for layer in model.bert.encoder.layer[-2:]:
            for param in layer.parameters():
                param.requires_grad = True

        optimizer = AdamW([
            {"params": model.bert.embeddings.parameters(), "lr": 1e-5},
            {"params": model.bert.encoder.layer[-2:].parameters(), "lr": 1e-5},
            {"params": model.pos_embedding.parameters(), "lr": 3e-4},
            {"params": model.decoder.parameters(), "lr": 3e-4},
        ], weight_decay=0.01)
        print("Разморозили embeddings + последние 2 слоя энкодера")

    model.train()
    epoch_train_loss, train_steps = 0.0, 0

    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        loss = train_step(
            model,
            input_ids,
            attention_mask,
            labels,
            optimizer,
            criterion,
            scaler=scaler,
            use_amp=use_amp,
        )
        epoch_train_loss += loss
        train_steps += 1
        pbar.set_postfix({"loss": f"{loss:.4f}"})

    avg_train = epoch_train_loss / train_steps
    train_losses.append(avg_train)

    model.eval()
    epoch_val_loss, val_steps = 0.0, 0

    for batch in tqdm(eval_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        loss = eval_step(model, input_ids, attention_mask, labels, criterion, use_amp=use_amp)
        epoch_val_loss += loss
        val_steps += 1

    avg_val = epoch_val_loss / val_steps
    val_losses.append(avg_val)

    print(f"Epoch {epoch+1}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), "best_model.pt")
        tokenizer.save_pretrained("./tokenizer")
        print(f"Лучшая модель (val_loss={avg_val:.4f})")

    if (epoch + 1) % 3 == 0:
        sample = next(iter(eval_dataloader))
        gen = model.generate(
            sample["input_ids"][:1].to(device),
            sample["attention_mask"][:1].to(device),
            tokenizer,
            strategy="beam",
            num_beams=6,
            temperature=0.8,
            min_len=24,
            repetition_penalty=1.8,
            max_len=56,
        )
        ref = tokenizer.decode(sample["labels"][0].tolist(), skip_special_tokens=True)

Epoch 1/8 [Val]: 100%|█████████████████████████████████████████████████████████████████| 40/40 [00:11<00:00,  3.38it/s]


Epoch 1: train_loss=9.3837, val_loss=9.1866
Лучшая модель (val_loss=9.1866)


Epoch 2/8 [Val]: 100%|█████████████████████████████████████████████████████████████████| 40/40 [00:11<00:00,  3.38it/s]


Epoch 2: train_loss=9.1788, val_loss=9.0931
Лучшая модель (val_loss=9.0931)


Epoch 3/8 [Val]: 100%|█████████████████████████████████████████████████████████████████| 40/40 [00:12<00:00,  3.25it/s]


...

Epoch 6: train_loss=8.4589, val_loss=8.7425


Epoch 7/8 [Val]: 100%|█████████████████████████████████████████████████████████████████| 40/40 [00:11<00:00,  3.34it/s]


Epoch 7: train_loss=8.4436, val_loss=8.7490


Epoch 8/8 [Val]: 100%|█████████████████████████████████████████████████████████████████| 40/40 [00:13<00:00,  3.06it/s]

Epoch 8: train_loss=8.4375, val_loss=8.7493


## Метрики качества

In [92]:
import evaluate

rouge_metric    = evaluate.load("rouge")
bleu_metric     = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")


def compute_metrics(predictions, references):
    results = {}
    # ROUGE
    rouge = rouge_metric.compute(predictions=predictions, references=references)
    results["rouge1"] = rouge["rouge1"]
    results["rouge2"] = rouge["rouge2"]
    results["rougeL"] = rouge["rougeL"]
    # BLEU
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )
    results["bleu"] = bleu["bleu"]
    # BERTScore
    bscore = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="ru",
        model_type="bert-base-multilingual-cased",
    )
    results["bertscore_f1"]        = float(np.mean(bscore["f1"]))
    results["bertscore_precision"] = float(np.mean(bscore["precision"]))
    results["bertscore_recall"]    = float(np.mean(bscore["recall"]))

    return results


def evaluation(model, dataloader, tokenizer, device, num_samples=50):
    model.eval()
    predictions, references = [], []
    count = 0

    for batch in tqdm(dataloader, desc="Evaluation"):
        if count >= num_samples:
            break

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"]

        for i in range(input_ids.size(0)):
            if count >= num_samples:
                break

            pred = model.generate(
                input_ids[i:i+1],
                attention_mask[i:i+1],
                tokenizer,
            )
            ref = tokenizer.decode(labels[i].tolist(), skip_special_tokens=True)

            predictions.append(pred)
            references.append(ref)
            count += 1

    metrics = compute_metrics(predictions, references)
    return metrics, predictions, references

## Обучение модели 

In [93]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model = model.to(device)

eval_data_sample = next(iter(eval_dataloader))

for i in range(min(3, eval_data_sample["input_ids"].size(0))):
    print(f"\n--- Пример {i+1} ---")
    original_text = tokenizer.decode(
        eval_data_sample["input_ids"][i].tolist(), skip_special_tokens=True
    )
    reference = tokenizer.decode(
        eval_data_sample["labels"][i].tolist(), skip_special_tokens=True
    )

    print(f"Текст (начало): {original_text[:200]}...")
    print(f"Эталон:         {reference[:200]}")

    for strategy in ["greedy", "top_k", "top_p", "beam"]:
        generated = model.generate(
            eval_data_sample["input_ids"][i:i+1].to(device),
            eval_data_sample["attention_mask"][i:i+1].to(device),
            tokenizer,
            strategy=strategy,
            max_len=64,
            top_k=40,
            top_p=0.92,
            temperature=0.8,
            min_len=24,
            repetition_penalty=1.8,
            num_beams=6,
        )
        print(f"  {strategy:>6s}: {generated[:200]}")



--- Пример 1 ---
Текст (начало): в 2020 году инфляция в россии составит 3, 5 - 4 %, прогнозирует центробанк. в первом квартале она ожидается так и вовсе ниже 3 %. впрочем, на ряд продовольственных товаров цены будут расти гораздо бол...
Эталон:         в уходящем году инфляция в россии находится на историческом минимуме. в следующем году ожидается, что она также будет минимальнои. однако стоимость ряда продуктов и напитков в 2020 году может вырасти 
  greedy: .,и в « » и на - с не россии что по зае а изаны о — к :
   top_k: . в на « »,и с по : что россии и ос вв изноир не виы
   top_p: .о в гон евро «и, слов что евцы удержать из нера для опаис милиционеры » активы -
    beam: .

--- Пример 2 ---
Текст (начало): глава белого дома дональд трамп выразил надежду на выполнение данных ему лидером кндр ким чен ыном обещании – так американскии президент отреагировал на заявление северокореиского политика о продолжен...
Эталон:         мировая общественность призвала лидера кндр ким чен ына не

## Реализация менее жадных стратегий выбора следующего токена (4 балла)
Всегда ли выбор наиболее вероятного токена на каждом шаге – это лучшая стратегия для генерации текста?

<details>
    <summary>Спойлер</summary>
    <p>Нет</p>
</details>

**Сравнение стратегий для генерации текста:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br><br/> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br><br/> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words <br/> that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br><br/> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps <br/> the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br><br/> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br><br/> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities <br/> by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br><br/> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br><br/> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br><br/> **Cons:** Computationally more complex and requires a good loss function. |

Ссылки на докуметацию:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

**1. Реализуйте стратегию Top-k в методе `generate`** (1 балл).   

**2. Реализуйте стратегию Nucleus Sampling (Top-p) в методе `generate`** (1 балл)

**3. Реализуйте стратегию Beam Search** (2 балла)

Получилось ли улучшить генерацию?

## Бонус (1 балл)

Что требуется сделать:

- от имеющейся модели "откусить" только декодерную часть
- написать цикл обучения (скорее поправить имеющийся) и дообучить декодер
- посмотреть качество генерации по метрикам и "глазами"
- ответить на вопрос "Дает ли применение Encoder-Decoder архитектуры значительный буст в качестве генерации?" с пруфами